# Phase B: Demucs Stem Separation (Steps 06a, 06b, 06c)

Runs Demucs stem separation and residual computation on Kaggle P100 GPU.  
Data pulled from HF Hub `Uday-4/djmix-v3`, results pushed back after every mix.  
Step 07 (CVXPY curve extraction) is excluded — run locally later via `hp_phase_b.py`.

**Nothing is deleted.** All stems and residuals stay on local disk and are uploaded to HF.

**Runtime:** P100 GPU (Settings → Accelerator → GPU P100).

**Usage:** Run cells 1-5 each session (upload this notebook to Kaggle). Resumable — skips completed mixes automatically.

In [ ]:
# Cell 1 — Install dependencies + clone repo
!pip install -q demucs huggingface_hub soundfile librosa cvxpy torchaudio ecos clarabel

import subprocess, sys
result = subprocess.run(
    ['git', 'clone', '--depth', '1', 'https://github.com/Uday-461/ai-dj-v4.git', '/kaggle/working/ai-dj/v4'],
    capture_output=True, text=True
)
if result.returncode != 0 and 'already exists' not in result.stderr:
    print('Clone failed:', result.stderr)
else:
    print('Repo ready at /kaggle/working/ai-dj/v4')

!pip install -q -e /kaggle/working/ai-dj/v4
sys.path.insert(0, '/kaggle/working/ai-dj/v4')

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Cell 2 — Config (EDIT THESE BEFORE RUNNING)
import os

HF_TOKEN  = "hf_..."           # <-- paste your HuggingFace token here
HF_REPO   = "Uday-4/djmix-v3"  # HF dataset repo
DATA_ROOT = "/kaggle/working/djmix"   # local working directory on Colab

# Colab free tier: 2 vCPUs for step 07 multiprocessing
CURVE_WORKERS = 2

# -------------------------------------------------------
os.makedirs(DATA_ROOT, exist_ok=True)
os.environ["AIDJ_DATA_ROOT"] = DATA_ROOT

# Validate token
if HF_TOKEN.startswith("hf_..."):
    raise ValueError("Edit Cell 2: set HF_TOKEN to your real HuggingFace token")

from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
try:
    api.whoami()
    print(f'HF auth OK — repo: {HF_REPO}')
except Exception as e:
    raise RuntimeError(f'HF auth failed: {e}')

In [ ]:
# Cell 3 — Load progress + manifest from HF
import json
import pickle
import shutil
import tempfile
from pathlib import Path
from huggingface_hub import hf_hub_download, list_repo_files

DATA_ROOT_PATH = Path(DATA_ROOT)


# --- Progress file ---
PROGRESS_KEY = "phase_b_progress.json"

def load_progress():
    """Download phase_b_progress.json from HF or create empty one."""
    try:
        local = hf_hub_download(
            repo_id=HF_REPO, filename=PROGRESS_KEY,
            repo_type="dataset", token=HF_TOKEN,
            local_dir=DATA_ROOT,
        )
        with open(local) as f:
            p = json.load(f)
        print(f'Loaded progress from HF: {PROGRESS_KEY}')
        return p
    except Exception:
        print('No progress file on HF — starting fresh')
        return {
            "stems_tracks":    [],
            "stems_segments":  [],
            "residuals":       [],
            "curves":          [],
        }


def save_progress_local(progress):
    path = DATA_ROOT_PATH / PROGRESS_KEY
    with open(path, 'w') as f:
        json.dump(progress, f, indent=2)
    return path


def push_progress(progress):
    """Upload progress JSON to HF immediately."""
    local_path = save_progress_local(progress)
    api.upload_file(
        path_or_fileobj=str(local_path),
        path_in_repo=PROGRESS_KEY,
        repo_id=HF_REPO,
        repo_type="dataset",
        token=HF_TOKEN,
        commit_message=f"Phase B progress update",
    )


# --- Manifest (transitions + alignments) ---
def download_manifest_files():
    """Download all transition and alignment PKLs from HF."""
    trans_dir  = DATA_ROOT_PATH / "results" / "transitions"
    align_dir  = DATA_ROOT_PATH / "results" / "alignments"
    trans_dir.mkdir(parents=True, exist_ok=True)
    align_dir.mkdir(parents=True, exist_ok=True)

    all_files = list(list_repo_files(repo_id=HF_REPO, repo_type="dataset", token=HF_TOKEN))

    tran_files  = [f for f in all_files if f.startswith("results/transitions/") and f.endswith(".pkl")]
    align_files = [f for f in all_files if f.startswith("results/alignments/")  and f.endswith(".pkl")]

    print(f'Found {len(tran_files)} transition PKLs, {len(align_files)} alignment PKLs on HF')

    for repo_path in tran_files + align_files:
        local_path = DATA_ROOT_PATH / repo_path
        if not local_path.exists():
            hf_hub_download(
                repo_id=HF_REPO, filename=repo_path,
                repo_type="dataset", token=HF_TOKEN,
                local_dir=DATA_ROOT,
            )

    return tran_files, align_files


def build_mix_list():
    """Build list of mixes from downloaded transition PKLs."""
    tran_dir = DATA_ROOT_PATH / "results" / "transitions"
    mixes = []
    for pkl_path in sorted(tran_dir.glob("*.pkl")):
        mix_id = pkl_path.stem
        # Skip old v2 data
        if mix_id == "mix0502":
            continue
        with open(pkl_path, 'rb') as f:
            transitions = pickle.load(f)
        # Collect unique track IDs
        track_ids = set()
        for t in transitions:
            if t.get("track_id_prev"): track_ids.add(t["track_id_prev"])
            if t.get("track_id_next"): track_ids.add(t["track_id_next"])
        mixes.append({
            "id": mix_id,
            "track_ids": sorted(track_ids),
            "transitions": transitions,
            "n_transitions": len(transitions),
        })
    return mixes


# --- Run ---
progress = load_progress()
tran_files, align_files = download_manifest_files()
all_mixes = build_mix_list()

print(f'\nMixes found: {len(all_mixes)}')
total_trans = sum(m['n_transitions'] for m in all_mixes)
print(f'Total transitions: {total_trans}')
print()
print(f'Progress (stems_tracks):   {len(progress["stems_tracks"])}/{len(all_mixes)} mixes done')
print(f'Progress (stems_segments): {len(progress["stems_segments"])}/{len(all_mixes)} mixes done')
print(f'Progress (residuals):      {len(progress["residuals"])}/{len(all_mixes)} mixes done')
print(f'Progress (curves):         {len(progress["curves"])}/{len(all_mixes)} mixes done')


# One-time fixup: remove mixes from curves progress if they have 0 curve files on HF
_hf_curves = set(f for f in list_repo_files(repo_id=HF_REPO, repo_type="dataset", token=HF_TOKEN)
                 if f.startswith("results/stem_curves/") and f.endswith(".npz"))
_mixes_with_curves = set(f.split("/")[2] for f in _hf_curves)
_bad = [m for m in progress["curves"] if m not in _mixes_with_curves]
if _bad:
    print(f'Removing mixes with 0 curves from progress: {_bad}')
    for m in _bad:
        progress["curves"].remove(m)
    push_progress(progress)

# Identify which mixes still need work
done_all = set(progress["stems_tracks"]) & set(progress["stems_segments"]) \
           & set(progress["residuals"]) & set(progress["curves"])
pending_mixes = [m for m in all_mixes if m["id"] not in done_all]
print(f'\nPending mixes (any step incomplete): {len(pending_mixes)}')
for m in pending_mixes:
    steps_done = []
    if m["id"] in progress["stems_tracks"]:   steps_done.append("06a")
    if m["id"] in progress["stems_segments"]: steps_done.append("06b_segs")
    if m["id"] in progress["residuals"]:      steps_done.append("06b_res")
    if m["id"] in progress["curves"]:         steps_done.append("07")
    print(f'  {m["id"]}: {m["n_transitions"]} transitions, done: {steps_done or ["nothing"]}')

In [ ]:
# Cell 4 — Main loop: 06a (track stems) + 06b (mix segment stems) + 06c (residuals)
# Step 07 (curve extraction) is intentionally excluded — run locally later.
# Nothing is deleted: all stems + residuals are uploaded to HF and kept on disk.
import sys
if '/kaggle/working/ai-dj/v4' not in sys.path:
    sys.path.insert(0, '/kaggle/working/ai-dj/v4')

import logging
import time
import shutil
import tempfile
import numpy as np
import librosa
from pathlib import Path
from huggingface_hub import hf_hub_download, list_repo_files

logging.basicConfig(level=logging.WARNING)

from aidj.stems.separator import StemSeparator
from aidj.stems.stem_cache import StemCache
from aidj.data.residual import compute_residual, align_track_to_mix_segment
from aidj import config


# -------------------------------------------------------
# Helpers
# -------------------------------------------------------

def download_track(tid):
    local = DATA_ROOT_PATH / "tracks" / f"{tid}.mp3"
    if local.exists():
        return local
    local.parent.mkdir(parents=True, exist_ok=True)
    hf_hub_download(
        repo_id=HF_REPO, filename=f"tracks/{tid}.mp3",
        repo_type="dataset", token=HF_TOKEN,
        local_dir=DATA_ROOT,
    )
    return local


def download_mix_audio(mix_id):
    local = DATA_ROOT_PATH / "mixes" / f"{mix_id}.mp3"
    if local.exists():
        return local
    local.parent.mkdir(parents=True, exist_ok=True)
    hf_hub_download(
        repo_id=HF_REPO, filename=f"mixes/{mix_id}.mp3",
        repo_type="dataset", token=HF_TOKEN,
        local_dir=DATA_ROOT,
    )
    return local


def upload_mix_outputs(mix, stem_cache):
    """Stage all stems + residuals for this mix into a temp dir and upload to HF."""
    mix_id = mix["id"]
    with tempfile.TemporaryDirectory() as staging_dir:
        staging = Path(staging_dir)
        n_files = 0

        def stage(src, rel):
            nonlocal n_files
            src = Path(src)
            if src.exists():
                dst = staging / rel
                dst.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(src, dst)
                n_files += 1

        # Track stems
        for tid in mix["track_ids"]:
            stem_dir = stem_cache._stem_dir("tracks", tid)
            if stem_dir.exists():
                for f in stem_dir.iterdir():
                    stage(f, f"stems/tracks/{tid}/{f.name}")

        # Mix segment stems
        for tran in mix["transitions"]:
            tran_id = tran["tran_id"]
            seg_dir = stem_cache._stem_dir("mix_segments", tran_id)
            if seg_dir.exists():
                for f in seg_dir.iterdir():
                    stage(f, f"stems/mix_segments/{tran_id}/{f.name}")

        # Residuals
        res_dir = DATA_ROOT_PATH / "results" / "residuals" / mix_id
        if res_dir.exists():
            for f in res_dir.iterdir():
                stage(f, f"results/residuals/{mix_id}/{f.name}")

        if n_files == 0:
            print(f'  [upload] No files to stage for {mix_id}')
            return

        print(f'  [upload] Uploading {n_files} files for {mix_id}...')
        api.upload_large_folder(
            folder_path=str(staging),
            repo_id=HF_REPO,
            repo_type="dataset",
        )
        print(f'  [upload] Done')


# -------------------------------------------------------
# Step 06a — Separate track stems
# -------------------------------------------------------

def step_06a_track_stems(mix, separator, stem_cache):
    mix_id = mix["id"]
    success = 0
    skipped = 0
    total = len(mix["track_ids"])

    for i, tid in enumerate(mix["track_ids"]):
        if stem_cache.has_stems("tracks", tid):
            skipped += 1
            continue
        try:
            track_path = download_track(tid)
            stem_cache.separate_and_cache(separator, str(track_path), "tracks", tid)
            success += 1
            print(f'  [06a] {mix_id} track {i+1}/{total} ({tid}): OK')
        except Exception as e:
            print(f'  [06a] {mix_id} track {i+1}/{total} ({tid}): FAILED — {e}')

    print(f'  [06a] {mix_id}: {success} separated, {skipped} cached, {total} total')
    return success + skipped, total


# -------------------------------------------------------
# Step 06b — Separate mix segments
# -------------------------------------------------------

def step_06b_mix_segments(mix, separator, stem_cache):
    mix_id = mix["id"]
    transitions = mix["transitions"]

    pending = [t for t in transitions
               if not stem_cache.has_stems("mix_segments", t["tran_id"])]
    if not pending:
        print(f'  [06b] {mix_id}: all {len(transitions)} segments already cached')
        return len(transitions), len(transitions)

    mix_path = download_mix_audio(mix_id)
    mix_audio, _ = librosa.load(str(mix_path), sr=config.SR, mono=True)

    success = 0
    skipped = len(transitions) - len(pending)

    for i, tran in enumerate(pending):
        tran_id = tran["tran_id"]
        mix_cue_in  = tran.get("mix_cue_in_time_next")
        mix_cue_out = tran.get("mix_cue_out_time_prev")
        if mix_cue_in is None or mix_cue_out is None:
            print(f'  [06b] {tran_id}: missing cue times, skip')
            continue

        mix_start = int(min(mix_cue_in, mix_cue_out) * config.SR)
        mix_end   = int(max(mix_cue_in, mix_cue_out) * config.SR)
        if mix_end - mix_start < config.SR:
            print(f'  [06b] {tran_id}: segment too short, skip')
            continue

        segment = mix_audio[mix_start:mix_end]
        try:
            stem_cache.separate_and_cache_segment(
                separator, segment, "mix_segments", tran_id
            )
            success += 1
            if success % 10 == 0:
                print(f'  [06b] {mix_id}: {success}/{len(pending)} segments done')
        except Exception as e:
            print(f'  [06b] {tran_id}: FAILED — {e}')

    print(f'  [06b] {mix_id}: {success} new, {skipped} cached, {len(transitions)} total')
    return success + skipped, len(transitions)


# -------------------------------------------------------
# Step 06c — Compute residuals
# -------------------------------------------------------

def step_06c_residuals(mix, stem_cache):
    mix_id = mix["id"]
    transitions = mix["transitions"]
    res_dir = DATA_ROOT_PATH / "results" / "residuals" / mix_id
    res_dir.mkdir(parents=True, exist_ok=True)

    success = 0
    skipped = 0

    for tran in transitions:
        tran_id  = tran["tran_id"]
        prev_id  = tran["track_id_prev"]
        next_id  = tran["track_id_next"]
        out_path = res_dir / f"{tran_id}.npz"

        if out_path.exists():
            skipped += 1
            continue

        mix_cue_in  = tran.get("mix_cue_in_time_next")
        mix_cue_out = tran.get("mix_cue_out_time_prev")
        if mix_cue_in is None or mix_cue_out is None:
            continue

        region_len = int((max(mix_cue_in, mix_cue_out) - min(mix_cue_in, mix_cue_out)) * config.SR)
        if region_len < config.SR:
            continue

        mix_seg_stems    = stem_cache.load_stems("mix_segments", tran_id)
        prev_track_stems = stem_cache.load_stems("tracks", prev_id)
        next_track_stems = stem_cache.load_stems("tracks", next_id)

        if mix_seg_stems is None:
            print(f'  [06c] {tran_id}: missing mix segment stems, skip')
            continue

        residual_data = {}
        if prev_track_stems is not None:
            t_start = tran.get("track_cue_in_time_prev", 0)
            aligned = {s: align_track_to_mix_segment(
                prev_track_stems[s], t_start, region_len, config.SR
            ) for s in config.STEMS if s in prev_track_stems}
            for stem, spec in compute_residual(mix_seg_stems, aligned).items():
                residual_data[f"{stem}_prev"] = spec

        if next_track_stems is not None:
            t_start = tran.get("track_cue_in_time_next", 0)
            aligned = {s: align_track_to_mix_segment(
                next_track_stems[s], t_start, region_len, config.SR
            ) for s in config.STEMS if s in next_track_stems}
            for stem, spec in compute_residual(mix_seg_stems, aligned).items():
                residual_data[f"{stem}_next"] = spec

        if residual_data:
            np.savez_compressed(str(out_path), **residual_data)
            success += 1

    print(f'  [06c] {mix_id}: {success} computed, {skipped} cached, {len(transitions)} total')
    return success + skipped, len(transitions)


# -------------------------------------------------------
# Main processing loop
# -------------------------------------------------------

separator  = StemSeparator(device="cuda" if torch.cuda.is_available() else "cpu")
stem_cache = StemCache(DATA_ROOT_PATH)
print(f'Separator device: {separator.device}')
print(f'Mixes to process: {len(pending_mixes)}')

session_start = time.time()

for mix_idx, mix in enumerate(pending_mixes):
    mix_id = mix["id"]
    mix_start_t = time.time()
    print(f'\n{"="*60}')
    print(f'[{mix_idx+1}/{len(pending_mixes)}] {mix_id} ({len(mix["track_ids"])} tracks, {mix["n_transitions"]} transitions)')
    print(f'{"="*60}')

    # Step 06a: track stems (GPU)
    if mix_id not in progress["stems_tracks"]:
        print(f'\n--- Step 06a: track stems ---')
        step_06a_track_stems(mix, separator, stem_cache)
        progress["stems_tracks"].append(mix_id)
        push_progress(progress)
    else:
        print(f'[06a] {mix_id}: already done, skipping')

    # Flush GPU cache between 06a and 06b to avoid OOM
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # Step 06b: mix segment stems (GPU)
    if mix_id not in progress["stems_segments"]:
        print(f'\n--- Step 06b: mix segment stems ---')
        step_06b_mix_segments(mix, separator, stem_cache)
        progress["stems_segments"].append(mix_id)
        push_progress(progress)
    else:
        print(f'[06b] {mix_id}: already done, skipping')

    # Step 06c: residuals (CPU)
    if mix_id not in progress["residuals"]:
        print(f'\n--- Step 06c: residuals ---')
        step_06c_residuals(mix, stem_cache)
        progress["residuals"].append(mix_id)
        push_progress(progress)
    else:
        print(f'[06c] {mix_id}: already done, skipping')

    # Upload stems + residuals to HF (nothing is deleted from local disk)
    print(f'\n--- Upload for {mix_id} ---')
    upload_mix_outputs(mix, stem_cache)

    elapsed = (time.time() - mix_start_t) / 60
    total_elapsed = (time.time() - session_start) / 3600
    print(f'\n[{mix_idx+1}/{len(pending_mixes)}] {mix_id} done in {elapsed:.1f} min | session total: {total_elapsed:.2f}h')

print(f'\n{"="*60}')
print(f'Phase B (stems + residuals) complete!')
print(f'  stems_tracks:   {len(progress["stems_tracks"])}/{len(all_mixes)} mixes')
print(f'  stems_segments: {len(progress["stems_segments"])}/{len(all_mixes)} mixes')
print(f'  residuals:      {len(progress["residuals"])}/{len(all_mixes)} mixes')
print(f'Step 07 (curves) will be run locally via hp_phase_b.py.')

In [ ]:
# Cell 5 — Verify results on HF
from huggingface_hub import list_repo_files
from collections import defaultdict

print('Counting files on HF Hub...')
all_hf_files = list(list_repo_files(
    repo_id=HF_REPO, repo_type="dataset", token=HF_TOKEN
))

# Count by folder
stems_tracks   = [f for f in all_hf_files if f.startswith("stems/tracks/")   and f.endswith(".ogg")]
stems_segments = [f for f in all_hf_files if f.startswith("stems/mix_segments/") and f.endswith(".ogg")]
residuals      = [f for f in all_hf_files if f.startswith("results/residuals/")  and f.endswith(".npz")]
curves         = [f for f in all_hf_files if f.startswith("results/stem_curves/") and f.endswith(".npz")]

# Per-mix breakdown
def per_mix_count(file_list, prefix):
    counts = defaultdict(int)
    for f in file_list:
        # e.g. "stems/tracks/track123/drums.ogg" -> extract mix from progress
        # or "results/residuals/mix0034/tran123.npz" -> mix_id = path parts[2]
        parts = f.split("/")
        if len(parts) >= 3:
            counts[parts[2]] += 1
    return counts

res_by_mix   = per_mix_count(residuals,  "results/residuals/")
crv_by_mix   = per_mix_count(curves,     "results/stem_curves/")

print(f'\nHF Hub summary:')
print(f'  stems/tracks/:        {len(stems_tracks)} OGG files')
print(f'  stems/mix_segments/:  {len(stems_segments)} OGG files')
print(f'  results/residuals/:   {len(residuals)} NPZ files')
print(f'  results/stem_curves/: {len(curves)} NPZ files')

print(f'\nPer-mix residuals:')
for mix_id in sorted(res_by_mix):
    print(f'  {mix_id}: {res_by_mix[mix_id]} residuals')

print(f'\nPer-mix stem curves:')
for mix_id in sorted(crv_by_mix):
    print(f'  {mix_id}: {crv_by_mix[mix_id]} curves')

print(f'\nProgress summary:')
for step, key in [("06a stems_tracks", "stems_tracks"), ("06b stems_segs", "stems_segments"),
                  ("06c residuals", "residuals"), ("07 curves", "curves")]:
    done = set(progress[key])
    total = len(all_mixes)
    print(f'  {step}: {len(done)}/{total} mixes — {sorted(done)}')